# Classify store products into a category

Which department should each newly listed product go into?
Each product belongs to exactly one of 14 categories, so this is multiclass classification: the model ranks every category and the top one is the prediction.

**Kumo Engine · Local GPU notebook · Synthetic data**

For products that carry several labels at once, see the [multilabel tagging example](local_kumo_engine_synthetic_product_tags.ipynb).
Run it on Linux with a supported NVIDIA GPU and local SSD scratch storage, as described in [Linux machine deployment](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment).
The full run takes one to two minutes on one NVIDIA L40S GPU.

## Install Kumo Engine

The text extra provides the word embeddings used to encode product titles.

In [1]:
%pip install "kumo-engine[text]"

Note: you may need to restart the kernel to use updated packages.


## Configure the example

Use the defaults to generate a small store catalog and a full `FAST` training run.
Each run creates its own directory under `OUTPUT_PARENT` for data, scratch files, the model and saved predictions; put it on a local SSD.
`SMOKE_MODE = True` limits training to a quick check that the workflow runs.

Set `KUMO_LOCAL_SSD` to an existing local SSD directory before starting the kernel. Each run creates its own subdirectory there.


In [2]:
import os

SEED = 7
N_BRANDS = 120
N_PRODUCTS = 3000
N_CUSTOMERS = 2000
N_PURCHASES = 30000
N_NEW_PRODUCTS = 200
N_NEW_PURCHASES = 2000
TOP_K = 3  # Categories returned per product.
OUTPUT_PARENT = os.environ["KUMO_LOCAL_SSD"]  # Existing local SSD directory.
SMOKE_MODE = False
CLEANUP = False

Import the tools and create this run's data and scratch directories.
Deterministic PyTorch operations let the reloaded model reproduce the saved scores on a GPU.

In [3]:
from pathlib import Path
from importlib.metadata import version
from importlib.util import find_spec
import os
import shutil
import tempfile

# cuBLAS reads this setting when PyTorch initializes CUDA.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
torch.use_deterministic_algorithms(True)

import numpy as np
import pandas as pd
from IPython.display import SVG, display
import kumo.engine as kumo
from kumoapi.model_plan import RunMode
from kumoapi.typing import Dtype, Stype

output_parent = Path(OUTPUT_PARENT).expanduser().resolve()
output_parent.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="product-category-", dir=output_parent))
data_dir = run_dir / "data"
data_dir.mkdir()
kumo.configure(scratch_dir=str(run_dir / "scratch"))
print(f"Engine {version('kumo-engine')} | results: {run_dir}")

/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Engine 1.0.1 | results: /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf


## Generate synthetic example data

Run the next cell to create an invented store catalog with brands, products, customers and purchases.
You can skim the generation code; the Kumo workflow starts at **Build the graph**.

| Table | Contents |
| --- | --- |
| `brands` | Invented brands with a price tier and an eco certification flag |
| `products` | Products with a brand, a title, a price and a shipping weight |
| `customers` | Invented customers with a shopper segment |
| `purchases` | Purchase lines linking a customer to a product |

Each product belongs to one of 14 categories and carries several storefront tags.
Brands specialize in a few categories and customers buy mostly from a few categories they prefer, so related products share brands and buyers.
Those specialties and preferences are hidden and never supplied as features.
Titles name the product type only part of the time, so the model also has to use brands, prices and purchases.
The same cell also reserves new products and their observed purchases for the later scoring batch.

In [4]:
rng = np.random.default_rng(SEED)
# Category: (storefront tags, product nouns, price range in dollars, weight range in kg)
CATALOG = {
    "laptops": (["electronics", "computing", "portable"], ["laptop", "notebook pc", "chromebook"], (300, 1800), (1.0, 2.8)),
    "smartphones": (["electronics", "mobile", "portable"], ["smartphone", "phone", "5g handset"], (120, 1300), (0.15, 0.25)),
    "headphones": (["electronics", "audio", "portable"], ["headphones", "earbuds", "headset"], (15, 400), (0.05, 0.4)),
    "televisions": (["electronics", "video", "home"], ["tv", "television", "smart display"], (180, 2500), (6.0, 30.0)),
    "printers": (["electronics", "computing", "office"], ["printer", "scanner", "all-in-one"], (60, 600), (4.0, 15.0)),
    "office supplies": (["office"], ["stapler", "binder", "pen set", "desk organizer"], (2, 45), (0.05, 1.5)),
    "kitchen appliances": (["home", "kitchen", "electronics"], ["blender", "toaster", "kettle", "stand mixer"], (20, 450), (1.0, 8.0)),
    "cookware": (["home", "kitchen"], ["skillet", "saucepan", "stock pot", "bakeware set"], (12, 250), (0.5, 6.0)),
    "toys": (["toys", "kids"], ["puzzle", "building blocks", "board game", "plush toy"], (6, 90), (0.1, 2.0)),
    "baby care": (["kids", "personal care"], ["diapers", "baby wipes", "bottle set", "baby lotion"], (4, 70), (0.2, 4.0)),
    "personal care": (["personal care"], ["shampoo", "toothbrush", "razor", "body lotion"], (3, 60), (0.1, 1.0)),
    "outdoor gear": (["outdoor", "sports", "portable"], ["tent", "hiking backpack", "camp lantern", "cooler"], (15, 450), (0.3, 9.0)),
    "fitness": (["sports", "health"], ["dumbbell set", "yoga mat", "resistance bands", "jump rope"], (8, 350), (0.2, 25.0)),
    "groceries": (["grocery"], ["coffee beans", "cereal", "pasta", "snack mix"], (2, 30), (0.2, 2.0)),
}
CATEGORIES = list(CATALOG)
ADJECTIVES = ["classic", "compact", "deluxe", "everyday", "pro", "smart", "ultra", "value"]
GENERIC_NOUNS = ["bundle", "essentials kit", "multipack", "set", "starter pack"]
SEGMENTS = ["family", "student", "professional", "outdoors", "home cook", "budget"]

brand_tier = rng.choice(["value", "standard", "premium"], N_BRANDS, p=[0.4, 0.4, 0.2])
brands = pd.DataFrame({
    "brand_id": [f"synthetic-brand-{i:03d}" for i in range(N_BRANDS)],
    "tier": brand_tier,
    "eco_certified": rng.random(N_BRANDS) < 0.25,
})
# Hidden: each brand makes products in two to four categories.
brand_categories = [rng.choice(len(CATEGORIES), rng.integers(2, 5), replace=False) for _ in range(N_BRANDS)]
tier_position = {"value": (0.0, 0.5), "standard": (0.25, 0.8), "premium": (0.6, 1.0)}

customers = pd.DataFrame({
    "customer_id": [f"synthetic-customer-{i:05d}" for i in range(N_CUSTOMERS)],
    "segment": rng.choice(SEGMENTS, N_CUSTOMERS),
})
# Hidden: each customer buys mostly from three preferred categories.
customer_categories = np.array([rng.choice(len(CATEGORIES), 3, replace=False) for _ in range(N_CUSTOMERS)])


def make_products(n, first_id):
    """Invent products; only the category and tags columns reveal their labels."""
    brand = rng.integers(0, N_BRANDS, n)
    category = np.array([rng.choice(brand_categories[b]) for b in brand])
    rows = []
    for b, c in zip(brand, category):
        tags, nouns, (low, high), (light, heavy) = CATALOG[CATEGORIES[c]]
        position = rng.uniform(*tier_position[brand_tier[b]])
        price = round(float(np.exp(np.log(low) + position * (np.log(high) - np.log(low)) + rng.normal(0, 0.4))), 2)
        noun = rng.choice(nouns) if rng.random() < 0.35 else rng.choice(GENERIC_NOUNS)
        product_tags = set(tags)
        if position > 0.8:
            product_tags.add("premium")
        if brands.eco_certified[b]:
            product_tags.add("eco friendly")
        rows.append({"brand_id": brands.brand_id[b],
                     "title": f"{rng.choice(ADJECTIVES)} {noun}",
                     "price": price,
                     "weight_kg": round(float(rng.uniform(light, heavy) * rng.lognormal(0, 0.4)), 2),
                     "category": CATEGORIES[c],
                     "tags": "|".join(sorted(product_tags))})
    frame = pd.DataFrame(rows)
    frame.insert(0, "product_id", [f"synthetic-product-{i:05d}" for i in range(first_id, first_id + n)])
    return frame


def make_purchases(product_frame, n, first_id):
    """Customers buy mostly from their preferred categories, sometimes from any category."""
    by_category = product_frame.groupby("category").product_id.agg(list)
    buyer = rng.integers(0, N_CUSTOMERS, n)
    product_ids = []
    for b in buyer:
        category = CATEGORIES[rng.choice(customer_categories[b])] if rng.random() < 0.8 else rng.choice(CATEGORIES)
        candidates = by_category.get(category) or product_frame.product_id.tolist()
        product_ids.append(rng.choice(candidates))
    return pd.DataFrame({
        "purchase_id": [f"synthetic-purchase-{i:06d}" for i in range(first_id, first_id + n)],
        "customer_id": customers.customer_id.to_numpy()[buyer],
        "product_id": product_ids,
        "quantity": rng.integers(1, 4, n),
    })


catalog = make_products(N_PRODUCTS, first_id=0)
purchases = make_purchases(catalog, N_PURCHASES, first_id=0)
display(pd.DataFrame({"rows": {"brands": len(brands), "products": len(catalog),
                               "customers": len(customers), "purchases": len(purchases)}}))
catalog.head()

# Reserve the new batch now; append it only after fitting.
new_products = make_products(N_NEW_PRODUCTS, first_id=N_PRODUCTS)
new_purchases = make_purchases(new_products, N_NEW_PURCHASES, first_id=N_PURCHASES)
simulated_labels = new_products[["product_id", "category"]]

,rows
brands,120
products,3000
customers,2000
purchases,30000


Keep the columns the model may see, with `category` as the label; `tags` belongs to the multilabel example.

In [5]:
LABELLED_COLUMNS = ["product_id", "brand_id", "title", "price", "weight_kg", "category"]
labelled_products = catalog[LABELLED_COLUMNS]
labelled_products.category.value_counts().to_frame("products")

,products
category,
personal care,293
printers,291
baby care,272
smartphones,229
headphones,220
groceries,212
toys,211
outdoor gear,202
fitness,193


Check that keys are unique and every purchase points to a known customer and product.

In [6]:
assert catalog.product_id.is_unique and brands.brand_id.is_unique and customers.customer_id.is_unique
assert purchases.purchase_id.is_unique
assert catalog.brand_id.isin(brands.brand_id).all()
assert purchases.customer_id.isin(customers.customer_id).all()
assert purchases.product_id.isin(catalog.product_id).all()
assert labelled_products.category.notna().all()
print("Keys and relationships are valid.")

Keys and relationships are valid.


## Build the graph

Save the tables as local Parquet and connect them with [Engine tables and graphs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs).
Titles are encoded as text; `brands` and `purchases` connect each product to related products through shared brands and shared buyers.
`category` is the prediction target, so the model never uses it as an input feature.
Keep the construction helper for the batch prediction step.

In [7]:
def save_tables(table_dir, product_rows, purchase_rows):
    table_dir.mkdir(exist_ok=True)
    product_rows.to_parquet(table_dir / "products.parquet", index=False)
    purchase_rows.to_parquet(table_dir / "purchases.parquet", index=False)
    brands.to_parquet(table_dir / "brands.parquet", index=False)
    customers.to_parquet(table_dir / "customers.parquet", index=False)


def build_graph(table_dir):
    files = kumo.FileConnector(root_dir=str(table_dir))
    products = (kumo.Table.from_source_table(files["products"], primary_key="product_id")
                .with_column("brand_id", stype=Stype.ID)
                .with_column("title", stype=Stype.text)
                .with_column("price", stype=Stype.numerical)
                .with_column("weight_kg", stype=Stype.numerical)
                .with_column("category", stype=Stype.categorical))
    brand_table = (kumo.Table.from_source_table(files["brands"], primary_key="brand_id")
                   .with_column("tier", stype=Stype.categorical)
                   .with_column("eco_certified", stype=Stype.categorical))
    customer_table = (kumo.Table.from_source_table(files["customers"], primary_key="customer_id")
                      .with_column("segment", stype=Stype.categorical))
    purchase_table = (kumo.Table.from_source_table(files["purchases"], primary_key="purchase_id")
                      .with_column("customer_id", stype=Stype.ID)
                      .with_column("product_id", stype=Stype.ID)
                      .with_column("quantity", stype=Stype.numerical))
    return kumo.Graph(
        tables={"products": products, "brands": brand_table,
                "customers": customer_table, "purchases": purchase_table},
        edges=[kumo.Edge("products", "brand_id", "brands"),
               kumo.Edge("purchases", "customer_id", "customers"),
               kumo.Edge("purchases", "product_id", "products")],
    )


save_tables(data_dir, labelled_products, purchases)
graph = build_graph(data_dir)
graph.validate()
print("Graph validated:", ", ".join(graph.tables))

Graph validated: products, brands, customers, purchases


Inspect the tables and relationships of the graph before training.
The relationship table always prints.
The SVG diagram follows [Visualize table relationships](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs#visualize-table-relationships) and needs the Graphviz Python package and the `dot` executable.

In [8]:
relationships = pd.DataFrame(
    [(edge.src_table, edge.fkey, edge.dst_table) for edge in graph.edges],
    columns=["table", "foreign_key", "references"],
)
display(relationships)

if shutil.which("dot") and find_spec("graphviz"):
    from graphviz import Digraph

    diagram = Digraph(format="svg")
    diagram.attr("node", shape="box")
    for name, table in graph.tables.items():
        diagram.node(name, label=f"{name}\nPK: {table.definition.pkey}")
    for edge in graph.edges:
        diagram.edge(edge.src_table, edge.dst_table, label=edge.fkey)
    display(SVG(diagram.pipe(format="svg")))
else:
    print("Install Graphviz and its `dot` executable to render the SVG diagram.")

,table,foreign_key,references
0,products,brand_id,brands
1,purchases,customer_id,customers
2,purchases,product_id,products


Install Graphviz and its `dot` executable to render the SVG diagram.


## Define the prediction question

Predict each product's category: `category` is a categorical column, so this is multiclass classification.
A static query fits because the label is a fixed product attribute rather than an outcome in a future time window, so Kumo assigns labelled products to training, validation and test splits at random.
See [Predictive queries](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries) for the syntax.

In [9]:
QUERY = "PREDICT products.category FOR EACH products.product_id"
query = kumo.PredictiveQuery(graph, QUERY)
query.validate()
assert query.get_task_type() == "multiclass_classification"
print(QUERY)
print("Task:", query.get_task_type())

PREDICT products.category FOR EACH products.product_id
Task: multiclass_classification


Materialize a snapshot of the graph and generate the training rows: one per labelled product.

In [10]:
snapshot = graph.snapshot()
training_table = query.generate_training_table(snapshot)
training_rows = pd.read_parquet(training_table.train_table_path)
print("Training rows:", len(training_rows))
training_rows.head()

ingest: table started phase="encode" table=products
ingest: table producers complete phase="encode" table=products rows=3000 elapsed_ms=113
ingest: Ingestion stage progress phase=Encode table=products shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=brands
ingest: table producers complete phase="encode" table=brands rows=120 elapsed_ms=2
ingest: Ingestion stage progress phase=Encode table=brands shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=customers
ingest: table producers complete phase="encode" table=customers rows=2000 elapsed_ms=24
ingest: Ingestion stage progress phase=Encode table=customers shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=purchases
ingest: table producers complete phase="encode" table=purchases rows=30000 elapsed_ms=328
ingest: Ingestion stage progress phase=Encode table=purchases shard=None progress_pct=100 current=1 total=1
ingest: encode 

Training rows: 3000


,ENTITY1,TIMESTAMP,TARGET1
0,synthetic-product-00234,1970-01-01,baby care
1,synthetic-product-00235,1970-01-01,personal care
2,synthetic-product-00236,1970-01-01,baby care
3,synthetic-product-00237,1970-01-01,smartphones
4,synthetic-product-00238,1970-01-01,cookware


## Train the model

Fit the suggested `FAST` plan and publish the model bundle to the run directory for the reload step.
Smoke mode caps epochs and steps only to check that the workflow runs.
See [Training](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training) for the available settings.

In [11]:
plan = query.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
if SMOKE_MODE:
    plan.optimization.max_epochs = 1
    plan.optimization.max_steps_per_epoch = 20

trainer = kumo.Trainer(plan)
model_store = kumo.FileConnector(root_dir=str(run_dir))
fit_result = trainer.fit(training_table, snapshot=snapshot,
                         export_config=kumo.ModelOutputConfig(model_store, path="model"))

Text column 'title' will not be encoded by default: its values are too short or have too few known words. To encode it anyway, set its encoder to TextEmbedding().
wrote col_stats.json (and kumo_col_stats/) under /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/scratch/snapshots/1a48efc71f592f84a99158b5db5f42f5e7bb4fd834b42a844296264b8f2f2c57/45fcc467d3833121d42e16f805418df16873e09d9c5a337a7a4df3299c52daaa/train_tables/.tt-791ba6538280492dbd1c44e1f84018d2-fit-59ba48fc7e964bfcba63e36637b3cdfe.ingested
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/kumo_core/automl/optuna_searcher.py:40: FutureWarning: `warn_independent_sampling` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  sampler=getattr(optuna.samplers, sampler_name)(**kwargs),
/mnt/localssd/kumo-notebook-execution-20261106/venv

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 0:   0%|          | 0/30 [00:00<?, ?it/s]

/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 0: 100%|██████████| 30/30 [00:01<00:00, 23.86it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 30/30 [00:01<00:00, 21.89it/s, val_loss=1.370, val_acc=0.526, train_loss=1.990, train_acc=0.343]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 30/30 [00:01<00:00, 22.72it/s, val_loss=1.040, val_acc=0.619, train_loss=1.110, train_acc=0.607]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 30/30 [00:01<00:00, 22.67it/s, val_loss=0.836, val_acc=0.702, train_loss=0.804, train_acc=0.717]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 30/30 [00:01<00:00, 22.53it/s, val_loss=0.692, val_acc=0.756, train_loss=0.582, train_acc=0.798]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?

... (22 outputs left out)

Restoring states from the checkpoint path at file:///mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/scratch/trainers/791ba6538280492dbd1c44e1f84018d2/runs/791ba6538280492dbd1c44e1f84018d2-fit-59ba48fc7e964bfcba63e36637b3cdfe/models/791ba6538280492dbd1c44e1f84018d2-fit-59ba48fc7e964bfcba63e36637b3cdfe/automl/version_0/checkpoint/epoch=6-step=210.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at file:///mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/scratch/trainers/791ba6538280492dbd1c44e1f84018d2/runs/791ba6538280492dbd1c44e1f84018d2-fit-59ba48fc7e964bfcba63e36637b3cdfe/models/791ba6538280492dbd1c44e1f84018d2-fit-59ba48fc7e964bfcba63e36637b3cdfe/automl/version_0/checkpoint/epoch=6-step=210.ckpt
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site

Testing DataLoader 0:   0%|          | 0/1 [00:00<?, ?it/s]

/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/kumo_core/explain/subgraph_collector.py:190: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['synthetic-brand-000' 'synthetic-brand-000' 'synthetic-brand-000' ...
 'synthetic-brand-119' 'synthetic-brand-119' 'synthetic-brand-119']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.


Testing DataLoader 0: 100%|██████████| 1/1 [00:06<00:00,  0.17it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            0.687889575958252
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


## Evaluate the model

Read the validation and test metrics from the fit.
`acc` is the share of products whose top category is correct.
These metrics come from synthetic data and are **demonstration only**.

In [12]:
split_metrics = {name: value for name, value in fit_result.metrics.items()
                 if name.startswith(("val_", "test_"))}
display(pd.Series(split_metrics, name="value").to_frame())

,value
val_loss,0.562034
val_acc,0.798077
test_loss,0.537311
test_acc,0.795666


## Batch predictions

### Inspect the reserved scoring data

New products arrive without a category.
The setup cell reserved these products and their observed purchases from existing customers using the same generators. Check their row counts and disjoint IDs before adding them to the graph.
Their simulated labels are kept aside for the comparison below and never enter the graph.

In [13]:
assert set(new_products.product_id).isdisjoint(catalog.product_id)
assert set(new_purchases.purchase_id).isdisjoint(purchases.purchase_id)
print(f"New products: {len(new_products)}; their purchases: {len(new_purchases)}")

New products: 200; their purchases: 2000


Add the new products with an empty `category` and their purchases to a separate copy of the tables, then refresh the graph snapshot.

In [14]:
scoring_dir = run_dir / "data_with_new_products"
unlabelled_new = new_products[LABELLED_COLUMNS].assign(category=None)
save_tables(scoring_dir,
            pd.concat([labelled_products, unlabelled_new], ignore_index=True),
            pd.concat([purchases, new_purchases], ignore_index=True))
scoring_graph = build_graph(scoring_dir)
scoring_graph.validate()
scoring_snapshot = scoring_graph.snapshot()
print("Products in the refreshed graph:", len(labelled_products) + len(unlabelled_new))

ingest: table started phase="encode" table=products
ingest: table producers complete phase="encode" table=products rows=3200 elapsed_ms=122
ingest: Ingestion stage progress phase=Encode table=products shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=brands
ingest: table producers complete phase="encode" table=brands rows=120 elapsed_ms=2
ingest: Ingestion stage progress phase=Encode table=brands shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=customers
ingest: table producers complete phase="encode" table=customers rows=2000 elapsed_ms=25
ingest: Ingestion stage progress phase=Encode table=customers shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=purchases
ingest: table producers complete phase="encode" table=purchases rows=32000 elapsed_ms=325
ingest: Ingestion stage progress phase=Encode table=purchases shard=None progress_pct=100 current=1 total=1
ingest: encode 

Products in the refreshed graph: 3200


A static query scores the products whose target is empty, so the [batch prediction population](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions) is exactly the new products.

In [15]:
scoring_query = kumo.PredictiveQuery(scoring_graph, QUERY)
population = scoring_query.generate_prediction_table(
    scoring_snapshot, plan=trainer.training_table_generation_plan)
scoring_rows = pd.read_parquet(population.table_path)
assert set(scoring_rows.ENTITY1) == set(new_products.product_id)
print("Products to score:", len(scoring_rows))

Products to score: 200


Score the new products, returning the `TOP_K` most likely categories for each.

In [16]:
predictions = trainer.predict(population, snapshot=scoring_snapshot, num_classes_to_return=TOP_K).frame
assert set(predictions.ENTITY) == set(new_products.product_id)
assert predictions.groupby("ENTITY").size().eq(TOP_K).all()
assert not predictions.duplicated(["ENTITY", "CLASS"]).any()
assert predictions.CLASS.isin(CATEGORIES).all()

wrote col_stats.json (and kumo_col_stats/) under /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/scratch/snapshots/fd530ffc80cfdb242a543af06a150b894299e40eaf7ff17c12131b607e9b0fd8/b3b98f97b5e0aa5457db3bd5d8fd4ad69f03c8cacfdbfd6c9c3a56cda2d53d2e/train_tables/.pred_b2a99ab44a3541b39ca34d4b40ac708c-791ba6538280492dbd1c44e1f84018d2-predict-a9fe8f9dbf05436ca02e3da40404845c.ingested
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at file:///mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/scratch/trainers/791ba6538

Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 52.15it/s]


### Save the predictions

`ENTITY` is the product, `CLASS` a candidate category and `SCORE` its probability; each product has `TOP_K` rows ranked by score.
Keep the top row as the predicted category, preview it next to the title, and save every candidate so a merchandiser can choose among them.

In [17]:
ranked = predictions.sort_values(["ENTITY", "SCORE"], ascending=[True, False], ignore_index=True)
top_category = (ranked.groupby("ENTITY").head(1)
                .rename(columns={"ENTITY": "product_id", "CLASS": "predicted_category", "SCORE": "probability"})
                [["product_id", "predicted_category", "probability"]])
display(top_category.merge(new_products[["product_id", "title", "price"]], on="product_id").head(10))

prediction_path = run_dir / "category_predictions.parquet"
ranked.rename(columns={"ENTITY": "product_id", "CLASS": "category", "SCORE": "probability"}
              ).to_parquet(prediction_path, index=False)
print(f"Saved {len(ranked)} rows for {ranked.ENTITY.nunique()} products to {prediction_path}")

,product_id,predicted_category,probability,title,price
0,synthetic-product-03000,groceries,0.905230,everyday desk organizer,7.51
1,synthetic-product-03001,personal care,0.781519,ultra shampoo,23.44
2,synthetic-product-03002,personal care,0.789678,deluxe essentials kit,6.64
3,synthetic-product-03003,personal care,0.462616,deluxe coffee beans,8.09
4,synthetic-product-03004,smartphones,0.906570,deluxe essentials kit,274.83
5,synthetic-product-03005,office supplies,0.880425,compact set,7.23
6,synthetic-product-03006,cookware,0.700381,pro set,11.14
7,synthetic-product-03007,printers,0.864109,compact all-in-one,444.25
8,synthetic-product-03008,smartphones,0.986474,smart multipack,464.87
9,synthetic-product-03009,fitness,0.830954,classic jump rope,164.41


Saved 600 rows for 200 products to /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/category_predictions.parquet


## Compare with the simulated categories

Check the predictions against the categories the simulation assigned to the new products.
Top-k accuracy counts a product as correct when its category is among the returned candidates; macro F1 weights every category equally.
The baseline predicts the most common catalog category for every product.
These synthetic-data metrics are **demonstration only**.

In [18]:
from sklearn.metrics import accuracy_score, f1_score

review = top_category.merge(simulated_labels, on="product_id", validate="one_to_one")
majority = labelled_products.category.mode()[0]
hits = ranked.merge(simulated_labels, left_on="ENTITY", right_on="product_id").assign(
    hit=lambda frame: frame.CLASS == frame.category)
display(pd.DataFrame({
    "Engine model": {"Accuracy": accuracy_score(review.category, review.predicted_category),
                     "Macro F1": f1_score(review.category, review.predicted_category, average="macro"),
                     f"Top-{TOP_K} accuracy": hits.groupby("ENTITY").hit.any().mean()},
    "Majority baseline": {"Accuracy": (review.category == majority).mean(),
                          "Macro F1": f1_score(review.category, [majority] * len(review), average="macro"),
                          f"Top-{TOP_K} accuracy": np.nan},
}).round(3))

,Engine model,Majority baseline
Accuracy,0.800,0.085
Macro F1,0.789,0.011
Top-3 accuracy,0.975,NaN


## Reload and score again

Restore the published bundle with [Publish and retrieve models](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models) and score the same new products without fitting again.
The reloaded model must return the same candidates, with scores within 0.0001 of the saved predictions.

In [19]:
reloaded = kumo.Trainer.load_from_connector(model_store, path="model")
rescored = reloaded.predict(population, snapshot=scoring_snapshot, num_classes_to_return=TOP_K).frame
key = ["ENTITY", "CLASS"]
original = predictions.sort_values(key).reset_index(drop=True)
restored = rescored.sort_values(key).reset_index(drop=True)
pd.testing.assert_frame_equal(original[key], restored[key])
max_difference = (original.SCORE - restored.SCORE).abs().max()
print("Reloaded scores:", len(restored), "| maximum absolute difference:", max_difference)
pd.testing.assert_series_equal(original.SCORE, restored.SCORE, rtol=0, atol=1e-4)

wrote col_stats.json (and kumo_col_stats/) under /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/scratch/snapshots/fd530ffc80cfdb242a543af06a150b894299e40eaf7ff17c12131b607e9b0fd8/b3b98f97b5e0aa5457db3bd5d8fd4ad69f03c8cacfdbfd6c9c3a56cda2d53d2e/train_tables/.pred_b2a99ab44a3541b39ca34d4b40ac708c-68e58533d9094e5baf5e6ca7556e6fa3-predict-94d1d499218e4ba7890e4f4304fa8721.ingested
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at file:///mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf/scratch/trainers/68e58533d

Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 47.62it/s]
Reloaded scores: 600 | maximum absolute difference: 0.0


## Keep or remove this run's files

`CLEANUP = False` keeps the generated tables, model bundle and predictions in the run directory.
Set `CLEANUP = True` to remove only the directory created by this notebook.

In [20]:
if CLEANUP:
    shutil.rmtree(run_dir)
print("Removed this run's files." if CLEANUP else f"Results retained in {run_dir}.")

Results retained in /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_product_category/20261106T223806Z/product-category-4uc37kkf.
